In [ ]:
import tensorflow as tf
import torch
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline

In [ ]:
import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))
# should return true
print(tf.test.is_built_with_cuda())
# will list your available gpu
print(tf.config.list_physical_devices('GPU'))

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

QLoRA = True
if QLoRA:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    # Increased rank and alpha for better quality on 24GB VRAM
    lora_config = LoraConfig(
        r=64,
        lora_alpha=128,
        target_modules="all-linear",
        bias="none",
        task_type="CAUSAL_LM",
        lora_dropout=0.05
    )
else:
    lora_config = None

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16 training

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    attn_implementation="flash_attention_2" # Use Flash Attention 2 if available (Ampere+)
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4, # Increased for 24GB VRAM
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_dir='./logs',
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="no",
    bf16=True, # Use bf16 for Ampere+ GPUs
    tf32=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    gradient_checkpointing=True,
    optim="paged_adamw_32bit" # Memory efficient optimizer
)

# Data Preparation: Creating the Multitask Dataset
To train our 3 specialized agents, we need to "decompose" the existing DeepMind Code Contests dataset.
The dataset gives us **Descriptions** (Narrative + Math) and **Solutions** (Code).
We need to generate the missing link: the **Math Basis**.

**Strategy:**
1.  Use the Base LLM to "reverse engineer" the **Math Basis** from the **Description**.
2.  Create 3 sub-datasets:
    *   **Math Agent**: `(Topic + Difficulty)` $\to$ `(Math Basis)`
    *   **Code Agent**: `(Math Basis)` $\to$ `(Solution Code)`
    *   **Narrative Agent**: `(Math Basis + Theme)` $\to$ `(Description)`

In [ ]:
from datasets import load_dataset, load_from_disk, Dataset
import os

# 1. Load Raw Dataset
# Check if local copy exists (from download_data.py)
if os.path.exists("dm-code_contests"):
    print("Loading from local disk (dm-code_contests/)...")
    full_dataset = load_from_disk("dm-code_contests")
    # Select a subset for demonstration
    dataset = full_dataset.select(range(50))
else:
    print("Local dataset not found. Downloading subset from Hugging Face Hub...")
    dataset = load_dataset("deepmind/code_contests", split="train[:50]") 

# 2. Define the "Reverse Engineering" Prompt
def create_extraction_prompt(description):
    return [
        {"role": "system", "content": "Extract the formal mathematical specification from this problem description. Remove all story elements. Keep constraints and input/output formats."},
        {"role": "user", "content": description}
    ]

# 3. Generate Synthetic "Math Basis" (This step takes time!)
# We use the loaded base model to process the dataset.
print("Generating synthetic Math Basis for training data...")

def add_math_basis(batch):
    # In a real run, we would batch inference here. 
    # For this demo, we'll simulate the output or run one-by-one if GPU allows.
    
    # Simulating the extraction for demonstration speed:
    # real_output = pipe(create_extraction_prompt(batch['description']))
    batch['math_basis'] = [f"Formal Spec of: {d[:50]}..." for d in batch['description']] 
    return batch

# Apply the transformation (Map)
processed_dataset = dataset.map(add_math_basis, batched=True, batch_size=4)

print("Dataset enriched with 'math_basis'.")

In [ ]:
# 4. Format Data for Each Agent (Task-Specific Formatting)

def format_for_multitask(example):
    output_texts = []
    
    # --- Data for Math Agent ---
    # Input: Metadata -> Output: Math Basis
    diff = example.get('difficulty', 1000)
    tags = ', '.join(example.get('tags', []))
    math_prompt = f"### Instruction:\n[Task: Math] Create a problem spec.\nTopic: {tags}\nDifficulty: {diff}\n\n### Response:\n{example['math_basis']}"
    output_texts.append(math_prompt)
    
    # --- Data for Code Agent ---
    # Input: Math Basis -> Output: Solution (Python)
    # We use the first python solution available
    sol = example['solutions']['solution'][0] if example['solutions'] and len(example['solutions']['solution']) > 0 else "print('No solution')"
    code_prompt = f"### Instruction:\n[Task: Code] Write a solution for:\n{example['math_basis']}\n\n### Response:\n```python\n{sol}\n```"
    output_texts.append(code_prompt)
    
    # --- Data for Narrative Agent ---
    # Input: Math Basis -> Output: Original Description
    narrative_prompt = f"### Instruction:\n[Task: Narrative] Write a story for:\n{example['math_basis']}\n\n### Response:\n{example['description']}"
    output_texts.append(narrative_prompt)
    
    return output_texts

# This function now returns 3x the examples (one for each task)
# You can filter these into separate datasets if training separate adapters.

In [ ]:
# 5. Train the Adapters
# To train separate adapters, you would filter the dataset and run SFTTrainer 3 times.

# Example: Training the "Math Agent" Adapter
math_dataset = processed_dataset.map(lambda x: {"text": f"### Instruction:\n[Task: Math] ...\n### Response:\n{x['math_basis']}"}) # Simplified for brevity

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=lora_config,
    train_dataset=processed_dataset, # Use the full multitask dataset or filtered subset
    formatting_func=format_for_multitask, # Use our multitask formatter
    data_collator=collator,
    max_seq_length=2048,
    packing=False
)

print("Starting training...")
# trainer.train() # Uncomment to run
# trainer.save_model("./models/math_expert") # Save as specific adapter

# Multi-Agent Generation Pipeline (LoRA Swapping)
To achieve high quality, we use a **Mixture of Experts (MoE)** approach using specialized LoRA adapters.
Instead of one generalist model, we employ three specialized agents:
1.  **Math Agent**: Fine-tuned on formal specifications (LaTeX, logic).
2.  **Code Agent**: Fine-tuned on test case generation (Python, edge cases).
3.  **Narrative Agent**: Fine-tuned on creative writing and storytelling.

We use `PEFT` to load one base model and dynamically swap adapters during inference to fit within 24GB VRAM.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 1. Load Base Model (Shared Backbone)
base_model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    # load_in_4bit=True # Uncomment for lower VRAM usage
)

# 2. Load Specialized Adapters
# In a real scenario, these would be paths to your fine-tuned checkpoints
# e.g., model = PeftModel.from_pretrained(base_model, "./models/math_expert", adapter_name="math")
# For this demo, we simulate by registering the same base weights as different adapters
# or just using the base model if adapters aren't trained yet.

try:
    model = PeftModel.from_pretrained(base_model, "./models/math_expert", adapter_name="math")
    model.load_adapter("./models/code_expert", adapter_name="code")
    model.load_adapter("./models/narrative_expert", adapter_name="narrative")
except Exception as e:
    print(f"Adapters not found ({e}), falling back to base model for demonstration.")
    model = base_model

def generate_with_adapter(prompt, adapter_name="default", max_new_tokens=512):
    # Switch to the specific expert adapter
    if hasattr(model, "set_adapter") and adapter_name in model.peft_config:
        model.set_adapter(adapter_name)
        print(f"Activated Adapter: {adapter_name}")
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    
    return tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print("Multi-Agent Pipeline Ready.")

In [ ]:
# Agent 1: The Mathematician (Math Adapter)
def agent_math_basis(topic, difficulty):
    prompt = f"""### Instruction:
Create a formal mathematical specification for a competitive programming problem.
Topic: {topic}
Difficulty: {difficulty}/3000

Output MUST strictly follow this format:
**Problem Name**: <name>
**Math Statement**: <formal definition of input and required output>
**Constraints**: <input bounds>
**Input Format**: <structure of input>
**Output Format**: <structure of output>

### Response:
"""
    return generate_with_adapter(prompt, adapter_name="math")

# Agent 2: The Engineer (Code Adapter)
def agent_test_generator(math_basis):
    prompt = f"""### Instruction:
Based on this problem specification, write a Python script to generate 10 valid test cases.
    
Specification:
{math_basis}

The script must:
1. Define a function `solve(input_str)` that implements the brute-force or correct solution.
2. Generate random inputs respecting the **Constraints**.
3. Print the input to `input_<i>.txt` and expected output to `output_<i>.txt`.
4. Be self-contained and executable.

### Response:
"""
    code = generate_with_adapter(prompt, adapter_name="code", max_new_tokens=1024)
    if "```python" in code:
        code = code.split("```python")[1].split("```")[0]
    return code

# Agent 3: The Storyteller (Narrative Adapter)
def agent_narrative(math_basis, theme="sci-fi"):
    prompt = f"""### Instruction:
Write a competitive programming problem description based on this mathematical specification.
    
Specification:
{math_basis}

Theme: {theme}

Instructions:
1. Wrap the math in an engaging story.
2. Clearly explain the input and output formats.
3. Include the constraints.
4. Provide 2 sample cases (Input and Output).

### Response:
"""
    return generate_with_adapter(prompt, adapter_name="narrative", max_new_tokens=1024)

In [ ]:
# Orchestrate the Collaboration
topic = "Graph Theory - Shortest Path"
difficulty = 1800

print(f"--- [Math Agent] Designing Core Logic for {topic} ---")
math_spec = agent_math_basis(topic, difficulty)
print(math_spec)

print("\n--- [Code Agent] Building Verification Script ---")
test_script = agent_test_generator(math_spec)
print(test_script)

# In a real loop, we would run the script here and feed errors back to the Code Agent
# if execution_failed:
#    agent_code_fix(error_log)

print("\n--- [Narrative Agent] Weaving the Story ---")
final_problem = agent_narrative(math_spec, theme="Space Exploration")
print(final_problem)